# Pythia-160M SAE: trening albo analiza

Ten notebook jest przeznaczony do uruchamiania w Kaggle. Wykonuje dokładnie jeden z dwóch głównych etapów: `TRAIN` albo `ANALYZE`. Nie zawiera treningu pilotowego. Jeśli przygotowane sekwencje nie są jeszcze dostępne, uruchamia automatycznie sekwencjonowanie wskazanego pliku The Pile do `/kaggle/working/pythia160m_sae_pilecc/sequenced/`.

In [ ]:
import os
from pathlib import Path
import shlex
import subprocess
import sys

# =========================
# JEDYNE USTAWIENIA SESJI
# =========================
RUN_MODE = "ANALYZE"  # "TRAIN" albo "ANALYZE"
VERBOSE = "low"       # "low" ogranicza logi Kaggle; "high" pokazuje tqdm
VERBOSE_INTERVAL = 1000          # kroki treningu albo chunki analizy
SEQUENCE_VERBOSE_INTERVAL = 1_000_000  # tokeny podczas sekwencjonowania

# None oznacza pełną analizę wszystkich przygotowanych sekwencji.
# Wartość całkowita uruchamia analizę tylko na próbce.
ANALYSIS_MAX_SEQUENCES = None
ANALYSIS_SAMPLING = "uniform"  # używane tylko, gdy ustawiono limit

CODE_DIR = Path("/kaggle/input/datasets/erykmikoajek/sae-training-and-moeffication")
MAIN = CODE_DIR / "main.py"
RESULTS_DIR = Path("/kaggle/working/pythia160m_sae_pilecc")
PILE_JSONL = Path("/kaggle/input/datasets/dschettler8845/the-pile-dataset-part-00-of-29/00.jsonl")
BEST_SAE = Path("/kaggle/input/datasets/erykmikoajek/trained-sae-models/topk_sae_layer_6_best.pt")
MODEL_NAME = "EleutherAI/pythia-160m"

LAYER_NUM = 6
SEQ_LENGTH = 256
MODEL_BATCH_SIZE = 2
CHUNK_SEQUENCES = 16
BATCH_SIZE_SAE = 1024
EXPANSION_FACTOR = 16
K = 64
NUM_EPOCHS = 2
TRAIN_RESUME = True
AUTO_SEQUENCE_IF_MISSING = True
SEQUENCE_MAX_TOKENS = None  # None = cały plik; np. 10_000_000 = limit testowy
SEQUENCE_MAX_DOCUMENTS = None

if RUN_MODE not in {"TRAIN", "ANALYZE"}:
    raise ValueError("RUN_MODE musi mieć wartość 'TRAIN' albo 'ANALYZE'")
if VERBOSE not in {"low", "high"}:
    raise ValueError("VERBOSE musi mieć wartość 'low' albo 'high'")
if VERBOSE_INTERVAL < 1:
    raise ValueError("VERBOSE_INTERVAL musi być dodatni")
if SEQUENCE_VERBOSE_INTERVAL < 1:
    raise ValueError("SEQUENCE_VERBOSE_INTERVAL musi być dodatni")
if ANALYSIS_MAX_SEQUENCES is not None and ANALYSIS_MAX_SEQUENCES < 1:
    raise ValueError("ANALYSIS_MAX_SEQUENCES musi być dodatni albo None")
if ANALYSIS_SAMPLING not in {"head", "tail", "uniform"}:
    raise ValueError("ANALYSIS_SAMPLING musi być: head, tail albo uniform")
if SEQUENCE_MAX_TOKENS is not None and SEQUENCE_MAX_TOKENS < 1:
    raise ValueError("SEQUENCE_MAX_TOKENS musi być dodatni albo None")
if SEQUENCE_MAX_DOCUMENTS is not None and SEQUENCE_MAX_DOCUMENTS < 1:
    raise ValueError("SEQUENCE_MAX_DOCUMENTS musi być dodatni albo None")
if not CODE_DIR.is_dir():
    raise FileNotFoundError(f"Brak katalogu ze źródłami: {CODE_DIR}")
if not MAIN.is_file():
    raise FileNotFoundError(f"Brak pliku main.py: {MAIN}")
required_sources = ["autoencoder_training.py", "activations_collecting.py", "features_analysis.py", "dataset_sequencing.py"]
missing_sources = [name for name in required_sources if not (CODE_DIR / name).is_file()]
if missing_sources:
    raise FileNotFoundError(f"Brak plików źródłowych w Kaggle dataset: {missing_sources}")
main_source = MAIN.read_text(encoding="utf-8")
if "os." in main_source and "import os" not in main_source:
    raise RuntimeError("main.py używa modułu os, ale go nie importuje. Zaktualizuj Kaggle dataset ze źródłami.")
source_markers = {
    "main.py": ["--verbose-interval"],
    "autoencoder_training.py": ["verbose_interval"],
    "features_analysis.py": ["verbose_interval"],
    "dataset_sequencing.py": ["root.text", "verbose_interval"],
}
for source_name, markers in source_markers.items():
    source_text = (CODE_DIR / source_name).read_text(encoding="utf-8")
    missing_markers = [marker for marker in markers if marker not in source_text]
    if missing_markers:
        raise RuntimeError(
            f"Nieaktualny {source_name}; brak elementów {missing_markers}. "
            "Ponownie opublikuj zaktualizowany dataset ze źródłami."
        )

print(f"RUN_MODE: {RUN_MODE}")
print(f"VERBOSE: {VERBOSE}, interval={VERBOSE_INTERVAL}")
print(f"Sequence verbose interval: {SEQUENCE_VERBOSE_INTERVAL:,} tokens")
print(f"Results/sequences: {RESULTS_DIR}")
print(f"Checkpoint: {BEST_SAE}")
print(f"Raw Pile JSONL (source reference): {PILE_JSONL}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', '<not set>')}")

In [ ]:
def run_cli(*args):
    command = [sys.executable, str(MAIN), *map(str, args)]
    print(shlex.join(command))
    subprocess.run(command, check=True)

# Szybki test wykrywa starą wersję plików .py zanim rozpocznie się długi etap.
help_probe = subprocess.run(
    [sys.executable, str(MAIN), "--help"],
    capture_output=True,
    text=True,
)
if help_probe.returncode != 0:
    raise RuntimeError(
        "Nie można uruchomić main.py --help. Szczegóły:\n"
        + help_probe.stdout
        + help_probe.stderr
    )
if "--verbose" not in help_probe.stdout:
    raise RuntimeError(
        "Kaggle dataset zawiera starą wersję kodu bez opcji --verbose. "
        "Ponownie opublikuj zaktualizowane pliki .py jako dataset Kaggle."
    )
print("CLI preflight: main.py oraz opcja --verbose są dostępne")

COMMON_ARGS = [
    "--profile", "local-50gb",
    "--model-name", MODEL_NAME,
    "--tokenizer-name", MODEL_NAME,
    "--data-path", str(RESULTS_DIR),
    "--layer-num", str(LAYER_NUM),
    "--seq-length", str(SEQ_LENGTH),
    "--model-batch-size", str(MODEL_BATCH_SIZE),
    "--chunk-sequences", str(CHUNK_SEQUENCES),
    "--batch-size-sae", str(BATCH_SIZE_SAE),
    "--expansion-factor", str(EXPANSION_FACTOR),
    "--k", str(K),
    "--num-epochs", str(NUM_EPOCHS),
    "--verbose", VERBOSE,
    "--verbose-interval", str(VERBOSE_INTERVAL),
    "--no-interactive",
]

sequence_tokens = RESULTS_DIR / "sequenced" / "tokens_seqs_padded.npy"
sequence_mask = RESULTS_DIR / "sequenced" / "attention_mask.npy"
missing_sequence_files = [
    path for path in (sequence_tokens, sequence_mask) if not path.is_file()
]
if missing_sequence_files:
    print("Brak przygotowanych sekwencji:")
    for path in missing_sequence_files:
        print(f"  - {path}")
    if not AUTO_SEQUENCE_IF_MISSING:
        raise FileNotFoundError(
            "AUTO_SEQUENCE_IF_MISSING=False, więc sekwencjonowanie nie zostanie uruchomione."
        )
    if not PILE_JSONL.is_file():
        raise FileNotFoundError(
            f"Nie znaleziono źródłowego datasetu The Pile: {PILE_JSONL}. "
            "Nie można automatycznie wykonać sekwencjonowania."
        )

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    sequence_args = [
        "--stage", "sequence",
        "--profile", "local-50gb",
        "--model-name", MODEL_NAME,
        "--tokenizer-name", MODEL_NAME,
        "--data-path", str(RESULTS_DIR),
        "--input-path", str(PILE_JSONL),
        "--seq-length", str(SEQ_LENGTH),
        "--min-seq-length", "10",
        "--batch-sentences", "32",
        "--verbose", VERBOSE,
        "--verbose-interval", str(SEQUENCE_VERBOSE_INTERVAL),
        "--no-interactive",
    ]
    if SEQUENCE_MAX_TOKENS is not None:
        sequence_args.extend(["--max-tokens", str(SEQUENCE_MAX_TOKENS)])
    if SEQUENCE_MAX_DOCUMENTS is not None:
        sequence_args.extend(["--max-documents", str(SEQUENCE_MAX_DOCUMENTS)])
    print("Uruchamiam automatyczne sekwencjonowanie The Pile...")
    run_cli(*sequence_args)

if not sequence_tokens.is_file() or not sequence_mask.is_file():
    raise RuntimeError(
        "Sekwencjonowanie zakończyło się, ale wymagane pliki nadal nie istnieją: "
        f"{sequence_tokens}, {sequence_mask}"
    )
print(f"Prepared tokens: {sequence_tokens}")
print(f"Attention mask: {sequence_mask if sequence_mask.exists() else 'not found'}")

## Trening SAE

Uruchomi się tylko wtedy, gdy `RUN_MODE = "TRAIN"`. Wznowienie korzysta z checkpointu znajdującego się w `RESULTS_DIR/models/checkpoints/`; zmiana `VERBOSE` nie zmienia konfiguracji modelu.

In [ ]:
if RUN_MODE == "TRAIN":
    train_args = ["--stage", "train-sae", *COMMON_ARGS]
    if not TRAIN_RESUME:
        train_args.append("--no-resume")
    run_cli(*train_args)
else:
    print("Pomijam trening: RUN_MODE=ANALYZE")

## Pełna analiza SAE

Domyślnie `ANALYSIS_MAX_SEQUENCES = None`, więc analizator przechodzi po całym `tokens_seqs_padded.npy`. Dzięki temu można porównać cechy obserwowane w analizie z `usage_counts` zapisanym podczas treningu i odróżnić cechy martwe od niewidzianych w danej próbce.

In [ ]:
if RUN_MODE == "ANALYZE":
    if not BEST_SAE.exists():
        raise FileNotFoundError(f"Brak checkpointu SAE: {BEST_SAE}")
    analyze_args = [
        "--stage", "analyze",
        *COMMON_ARGS,
        "--checkpoint-path", str(BEST_SAE),
    ]
    if ANALYSIS_MAX_SEQUENCES is not None:
        analyze_args.extend([
            "--max-sequences", str(ANALYSIS_MAX_SEQUENCES),
            "--analysis-sampling", ANALYSIS_SAMPLING,
        ])
    run_cli(*analyze_args)
else:
    print("Pomijam analizę: RUN_MODE=TRAIN")

In [ ]:
print("Gotowe. Wyniki i logi analizy/treningu są w: " + str(RESULTS_DIR))
print("Analiza tekstowa: " + str(RESULTS_DIR / 'analysis' / 'features_analysis.txt'))
print("Analiza JSON: " + str(RESULTS_DIR / 'analysis' / 'features_analysis.json'))